In [1]:
!source ../miniconda3/bin/activate
!conda --version

conda 24.11.3


In [5]:
import os
import json
import pandas as pd
import torch
from PIL import Image
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

# --------------------------------------------------
# Configuration
# --------------------------------------------------
MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"
IMAGE_DIR = "fire_dataset_2_augmented"
OUTPUT_CSV = "fire_captions_2.csv"
MAX_LENGTH = 512

In [3]:
# --------------------------------------------------
# Load processor and model
# --------------------------------------------------
# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    print("Warning: Running on CPU may be slow. GPU is recommended.")

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    device_map="auto",                 # 🧠 This spreads across GPUs
    torch_dtype=torch.bfloat16,         # Use bfloat16 for lower memory
)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
import re, json
# Directory of images
records = []

# Fire analysis prompt
system_prompt = (
    "You are a visual analyst evaluating an image for signs of fire and the surrounding context. "
    "Do the following tasks:\n"
    "1: Summarize what you see in the image. Describe the environment, key objects, people, and any signs of fire or smoke.\n"
    "2: Based on your summary, classify the fire situation: "
    "no fire(eg:, fire alarm, fire distinguisher,..), controlled fire (e.g., fireplace emitting, campfire, cooking, candles,..) or a dangerous/uncontrolled fire (e.g., curtains on fire, smoke on ceiling, couch on fire, bed sheet on fire,..)?\n"
    "Return only this JSON format:\n"
    "{ \"caption\": \"...\", \"label\": \"no fire\"|\"controlled fire\"|\"dangerous fire\" }"
)

# Loop through image files
for fname in os.listdir(IMAGE_DIR):
    if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue
    image_path = os.path.join(IMAGE_DIR, fname)
    try:
        image = Image.open(image_path).convert("RGB")

        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": system_prompt}
            ]
        }]

        # Generate prompt and inputs
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[image], return_tensors="pt")

        # Run model
        generate_ids = model.generate(**inputs, max_length=5000)
        output = processor.tokenizer.batch_decode(generate_ids, skip_special_tokens=True)[0]
        if output.startswith("system"):
            output = output.split("assistant", 1)[-1].strip()
        # Clean up any system/user/assistant roles
        clean_output = re.sub(r'^(system|user|assistant)\n', '', output, flags=re.MULTILINE).strip()

        # Match the last complete JSON block
        matches = re.findall(r'\{[^{}]+\}', clean_output, re.DOTALL)
        if matches:
            try:
                result = json.loads(matches[-1])  # take the last match (most likely correct)
                caption = result.get("caption", "").strip()
                label = result.get("label", "").strip()
            except Exception as e:
                print(f"❌ Failed to parse JSON for {fname}: {e}")
                caption = clean_output
                label = "unknown"
        else:
            print(f"❌ No JSON block found in output for {fname}")
            caption = clean_output
            label = "unknown"

        print(f"✅ {fname} - {label} - {caption} ")
        records.append({
            "image_path": image_path,
            "caption": caption,
            "label": label
        })

    except Exception as e:
        print(f"❌ Error processing {fname}: {e}")

/home/student4/miniconda3/lib/python3.12/site-packages/transformers/generation/utils.py:2347: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


✅ image1137_aug_1.jpg - dangerous fire - The image shows a living room with a table on fire. There is smoke rising from the flames, and the room appears to be well-lit. 
✅ image1123_aug_5.jpg - dangerous fire - The image shows a building engulfed in flames with thick smoke rising from the structure. The fire appears to be intense, with bright orange and yellow flames visible. 
✅ image1128_aug_5.jpg - dangerous fire - The image shows a close-up view of a fire with flames and smoke visible. The fire appears to be burning in a confined space, possibly a fireplace or a similar appliance. 
✅ image1025_aug_6.jpg - dangerous fire - The image shows a kitchen with flames coming from the oven and microwave, indicating a dangerous fire situation. 
✅ image1088_aug_4.jpg - dangerous fire - The image shows a silhouette of a chair against a backdrop of intense flames, suggesting a fire scenario. 
✅ image1028_aug_1.jpg - dangerous fire - The image shows a bedroom with a bed that appears to be on fire.

In [7]:
# Save results to CSV
df = pd.DataFrame(records)
df.to_csv("fire_image_captions_v2.csv", index=False)
print("📁 Saved results to fire_image_captions_v2.csv")

📁 Saved results to fire_image_captions_v2.csv
